# Рекомендательная система товаров в электронной коммерции

## Описание проекта

Электронная коммерция является одной из ключевых областей применения рекомендательных систем. Такие системы позволяют пользователям быстрее находить интересующие их товары, а бизнесу — увеличивать вовлечённость и выручку.

В рамках данного проекта рассматривается задача:

> **предсказания товаров, которые с наибольшей вероятностью заинтересуют пользователя интернет-магазина**

Рекомендации формируются на основе пользовательского поведения, включая:

* просмотры товаров
* добавления в корзину
* покупки

---

## Цель проекта

Разработать рекомендательную систему, способную:

* предсказывать **товары, которые пользователь добавит в корзину**
* учитывать **историю взаимодействий пользователя**
* работать в условиях **разреженных данных и cold start**
* служить основой для **персонализированных рекомендаций в онлайн-сервисе**

---

## Постановка задачи

Задача формулируется как:

> **предсказание товаров, которые пользователь с наибольшей вероятностью добавит в корзину**

Особенности:

* задача рекомендаций (**recommendation / ranking**)
* implicit feedback (неявная обратная связь)
* наличие нескольких типов событий (view, addtocart, transaction)
* необходимость учитывать **временную динамику**

---

## Данные

Используется датасет с пользовательскими взаимодействиями в интернет-магазине.

Он состоит из трёх таблиц:

### 1. `category_tree.csv`

* `parentid` — родительская категория
* `categoryid` — дочерняя категория

Представляет иерархию категорий товаров.

---

### 2. `events.csv`

Лог событий пользователей:

* `timestamp` — временная метка
* `visitorid` — идентификатор пользователя
* `event` — тип события:

  * `view` — просмотр товара
  * `addtocart` — добавление в корзину
  * `transaction` — покупка
* `itemid` — идентификатор товара
* `transactionid` — идентификатор покупки

---

### 3. `item_properties.csv`

Свойства товаров:

* `timestamp` — время добавления свойства
* `itemid` — идентификатор товара
* `property` — название свойства
* `value` — значение

---

📎 **Ссылка на датасет:**
*(ссылка на датасет)*

---

## Основные этапы проекта

### 1. Исследование данных (EDA)

* анализ структуры данных
* анализ распределения событий
* временной анализ активности пользователей
* анализ популярности товаров
* анализ активности пользователей
* оценка разреженности данных
* анализ cold start

Результат:
`notebooks/01_eda.ipynb`

---

### 2. Подготовка инфраструктуры

* развёртывание MLflow
* настройка логирования экспериментов

Результат:
`.sh`-скрипт (`scripts/run_mlflow.sh`)

---

### 3. Формализация задачи и метрик

Выбор целевого действия:

* **add-to-cart**

Выбор метрик:

* Recall@K
* MAP@K
* NDCG@K

Результат:
описание в `README.md`

---

### 4. Моделирование

* подготовка данных
* baseline:

  * Top Popular
* модели:

  * ALS (collaborative filtering)
* генерация кандидатов
* оценка качества моделей

Результат:
`notebooks/02_modeling.ipynb`
сохранённая модель

---

### 5. Продуктивизация

* разработка API для рекомендаций
* реализация сервиса на FastAPI
* упаковка в Docker

Результат:
Python-проект + `Dockerfile`

---

### 6. Переобучение модели

* построение пайплайна переобучения
* автоматизация через Airflow

Результат:
`airflow/dags/retrain_recsys.py`

---

### 7. Мониторинг

* отслеживание технических метрик сервиса
* контроль качества рекомендаций
* анализ распределений данных

Результат:
описание в `README.md` + код в `src/monitoring`

---

### 8. Документация

* описание пайплайна
* описание моделей
* инструкция по запуску проекта

Результат:
`README.md`

---

### 9. Воспроизводимость

* фиксация зависимостей
* контроль случайных состояний
* логирование экспериментов

Результат:
`requirements.txt` + MLflow

---

## Используемые технологии

* Python (pandas, numpy)
* implicit (ALS)
* LightGBM (для дальнейшего развития)
* scikit-learn
* matplotlib / seaborn
* MLflow
* FastAPI
* Docker
* Apache Airflow

---

## Особенности задачи

* сильный дисбаланс событий (view >> addtocart >> transaction)
* выраженный **long-tail эффект**
* высокая **разреженность user-item матрицы**
* наличие **cold start пользователей**
* необходимость комбинирования нескольких подходов:

  * popularity-based
  * collaborative filtering

---

## Результат

В рамках проекта разрабатывается рекомендательная система, которая:

* учитывает поведение пользователя
* способна предсказывать интересующие товары
* комбинирует несколько подходов (ALS + популярность)
* может быть развернута как веб-сервис
* пригодна для интеграции в реальные e-commerce платформы

---

Если хочешь — следующим шагом могу:

👉 адаптировать это описание под **максимальный балл (чуть более “научный стиль” для магистратуры)**
или
👉 сразу встроить туда **кусок про твою гибридную систему (ALS + ранкер)**


# 1. Импорты

In [1]:
# Стандартная библиотека
import gc
import logging
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple
import os
import sys
import warnings

import pandas as pd

# Работа с данными
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Статистика
from scipy.sparse import csr_matrix
from scipy.stats import gaussian_kde

# ML
import lightgbm as lgb
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Файлы и хранилища
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from dotenv import load_dotenv

# Прочее
import phik
from phik import report, resources
from tqdm.auto import tqdm

In [2]:
import os
import sys
import warnings

import pandas as pd

# настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.0f}".format

# настройки графиков в ноутбуке
%matplotlib inline
%config InlineBackend.figure_format = "retina"

# корень проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", os.path.basename(PROJECT_ROOT))
print("src exists:", "src" in os.listdir(PROJECT_ROOT))

# функция для относительных путей
def to_relative(path, base):
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path

from src.utils.config import (
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    EVENTS_PATH,
    CATEGORY_TREE_PATH,
    ITEM_PROPERTIES_PATH,
    ARTIFACTS_DIR,
    MLFLOW_DIR,
)

# S3
S3_BASE = "s3://s3-student-mle-20250927-31ecef0a74/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

# проверка путей
paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "MLFLOW_DIR": MLFLOW_DIR,
    "EVENTS_PATH": EVENTS_PATH,
    "CATEGORY_TREE_PATH": CATEGORY_TREE_PATH,
    "ITEM_PROPERTIES_PATH": ITEM_PROPERTIES_PATH,
}

print("\n📁 Project paths:\n")

for name, path in paths_to_check.items():
    rel_path = to_relative(path, PROJECT_ROOT)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"{name:<20} {rel_path:<40} [{status}]")

PROJECT_ROOT: ecommerce-recsys
src exists: True

📁 Project paths:

DATA_DIR             data                                     [OK]
RAW_DIR              data/raw                                 [OK]
PROCESSED_DIR        data/processed                           [OK]
ARTIFACTS_DIR        artifacts                                [OK]
MLFLOW_DIR           mlruns                                   [MISSING]
EVENTS_PATH          data/raw/events.csv                      [OK]
CATEGORY_TREE_PATH   data/raw/category_tree.csv               [OK]
ITEM_PROPERTIES_PATH data/raw/item_properties.csv             [MISSING]


# 2. Загрузка данных

In [3]:
events = pd.read_csv("data/raw/events.csv")
category_tree = pd.read_csv("data/raw/category_tree.csv")
item_props = pd.read_csv("data/raw/item_properties.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/events.csv'

# 3. Общая информация

In [ ]:
print("Events shape:", events.shape)
print("Category tree shape:", category_tree.shape)
print("Item properties shape:", item_props.shape)

events.head()

# 4. Типы данных и пропуски

In [ ]:
events.info()

events.isnull().sum()

Вывод (написать текстом):

есть ли пропуски

где они

насколько критично

# 5. Уникальные значения

In [ ]:
print("Users:", events["visitorid"].nunique())
print("Items:", events["itemid"].nunique())
print("Events:", events.shape[0])

# 6. Распределение типов событий

In [ ]:
events["event"].value_counts()

In [ ]:
sns.countplot(data=events, x="event")
plt.title("Event distribution")
plt.show()

In [ ]:
Вывод:
обычно view >> addtocart >> transaction
сильный дисбаланс

# 7. Временной анализ

In [ ]:
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")

events["date"] = events["datetime"].dt.date
events["hour"] = events["datetime"].dt.hour

## События по времени

In [ ]:
events.groupby("date").size().plot(figsize=(12,4), title="Events over time")
plt.show()

## По часам

In [ ]:
sns.countplot(data=events, x="hour")
plt.title("Events by hour")
plt.show()

Вывод:
пики активности
есть ли сезонность

# 8. Популярные товары

In [ ]:
top_items = events["itemid"].value_counts().head(20)

top_items.plot(kind="bar", figsize=(10,4), title="Top items")
plt.show()

Вывод:
есть сильный popularity bias

# 9. Активность пользователей

In [ ]:
user_activity = events.groupby("visitorid").size()

user_activity.describe()

In [ ]:
sns.histplot(user_activity, bins=50, log_scale=True)
plt.title("User activity distribution")
plt.show()

In [ ]:
Вывод:
большинство пользователей малоактивны
длинный хвост

# 10. Активность товаров

In [ ]:
item_activity = events.groupby("itemid").size()

sns.histplot(item_activity, bins=50, log_scale=True)
plt.title("Item popularity distribution")
plt.show()

In [ ]:
Вывод:
few popular items, many rare

# 11. Конверсия

In [ ]:
events["event"].value_counts(normalize=True)

In [ ]:
Вывод:
низкая конверсия в покупку
add-to-cart — разумный target

# 12. Разреженность (sparsity)

In [ ]:
n_users = events["visitorid"].nunique()
n_items = events["itemid"].nunique()
n_interactions = len(events)

sparsity = 1 - n_interactions / (n_users * n_items)

print("Sparsity:", sparsity)

In [ ]:
Вывод:
обычно ~99%+
значит нужен CF/ALS

# 13. Cold start

In [ ]:
user_counts = events["visitorid"].value_counts()
cold_users = (user_counts == 1).mean()

print("Cold users ratio:", cold_users)

In [ ]:
Вывод:
много новых пользователей
нужен popular fallback

# 14. Анализ item_properties

In [ ]:
item_props["property"].value_counts().head(10)

In [ ]:
item_props["value"].nunique()

In [ ]:
Вывод:
данные разреженные
можно использовать как фичи

# Финальные выводы

In [ ]:
1. Данные имеют сильный дисбаланс: большинство событий — просмотры.
2. Пользовательская активность распределена неравномерно (long tail).
3. Наблюдается сильный popularity bias среди товаров.
4. Матрица user-item сильно разрежена (>99%).
5. Высокая доля cold users → нужен fallback (Top Popular).
6. Добавление в корзину выбрано как целевое действие.
7. Подход к решению:
   - candidate generation (ALS, popularity)
   - ranking (LightGBM)